In [ ]:
import pandas as pd

# Since the file is right next to the notebook, we just use the file name
df = pd.read_csv('bank-additional-full.csv', sep=';')

# Verify it loaded correctly
print(df.head())

# Check your main conversion column
print("\nConversion Breakdown:")
print(df['y'].value_counts())

In [ ]:
# 1. Convert the target variable 'y' into a binary integer (1 or 0) for easier math later
df['converted'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# 2. Define the Funnel Stages
# Stage 1: Total Prospects Contacted (Everyone in the dataset)
total_prospects = len(df)

# Stage 2: Engaged Prospects
# Assumption: If the call duration is greater than 60 seconds, the user stayed to listen to the pitch.
engaged_prospects = len(df[df['duration'] > 60])

# Stage 3: Converted Customers (Users who subscribed to the deposit)
converted_customers = df['converted'].sum()

# 3. Print the Funnel Drop-off Metrics
print("--- Marketing Funnel Metrics ---")
print(f"Stage 1 - Total Prospects Contacted: {total_prospects}")

engaged_rate = (engaged_prospects / total_prospects) * 100
print(f"Stage 2 - Engaged (>60s call):       {engaged_prospects} ({engaged_rate:.1f}% of total)")

close_rate = (converted_customers / engaged_prospects) * 100
print(f"Stage 3 - Converted Customers:       {converted_customers} ({close_rate:.1f}% of engaged)")

In [ ]:
# Filter down to ONLY the users who engaged (stayed on the phone > 60s)
df_engaged = df[df['duration'] > 60].copy()

# Group by the 'job' column and calculate the conversion rate and total volume
job_conversion = df_engaged.groupby('job')['converted'].agg(['count', 'mean']).reset_index()

# Rename columns for clarity and convert the mean to a percentage
job_conversion.columns = ['Job Category', 'Total Engaged', 'Conversion Rate (%)']
job_conversion['Conversion Rate (%)'] = (job_conversion['Conversion Rate (%)'] * 100).round(1)

# Sort the results by the highest conversion rate
job_conversion = job_conversion.sort_values(by='Conversion Rate (%)', ascending=False)

# Display the findings
print("--- Conversion Rates by Job Category (Engaged Users) ---")
print(job_conversion.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the figure size and style
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Create a horizontal bar chart using the job_conversion dataframe
# We use a color palette to make it look professional
ax = sns.barplot(
    x="Conversion Rate (%)", 
    y="Job Category", 
    data=job_conversion, 
    palette="viridis",
    hue="Job Category", # Adding hue to avoid future seaborn warnings
    legend=False
)

# Add clear titles and labels
plt.title("Conversion Rate by Job Category (Engaged Users)", fontsize=14, weight='bold', pad=15)
plt.xlabel("Conversion Rate (%)", fontsize=12)
plt.ylabel("", fontsize=12) # Leave y-label blank since category names are obvious

# Attach the exact percentage to the end of each bar for easy reading
for container in ax.containers:
    ax.bar_label(container, padding=5, fmt='%.1f%%')

# Adjust layout and display the chart
plt.tight_layout()
plt.show()

In [ ]:
# List of the 3 key variables we want to analyze
variables_to_check = ['education', 'month', 'poutcome']

# Loop through each variable, calculate the metrics, and print a clean table
for var in variables_to_check:
    # Group by the variable and calculate counts and mean (conversion rate)
    insight_df = df_engaged.groupby(var)['converted'].agg(
        Total_Engaged='count', 
        Conversion_Rate='mean'
    ).reset_index()
    
    # Format the conversion rate as a clean percentage
    insight_df['Conversion_Rate (%)'] = (insight_df['Conversion_Rate'] * 100).round(1)
    
    # Drop the raw decimal column and sort by highest conversion
    insight_df = insight_df.drop(columns=['Conversion_Rate']).sort_values(by='Conversion_Rate (%)', ascending=False)
    
    # Print the formatted output
    print(f"\n--- Conversion Rates by {var.upper()} ---")
    print(insight_df.to_string(index=False))

In [ ]:
# Create a final, clean dataset for Power BI
# We will keep the new 'converted' column we engineered
df_export = df.copy()

# Save it as a standard CSV in your project folder
df_export.to_csv('powerbi_bank_funnel_data.csv', index=False)

print("Data exported successfully! Look for 'powerbi_bank_funnel_data.csv' in your VS Code explorer.")